In [1]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm

/home/fxr/Documents/personal/musica_arg_nlp/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-04-17 23:12:00.087466: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744942320.112153   15403 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744942320.120060   15403 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1744942320.134469   15403 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same t

In [2]:
df = pd.read_csv("../data/raw/100canciones_rock_argentino.csv")
print(f"{len(df)} rows")
df.head(3)

100 rows


,Posición,Título,Artista,Año,lyrics
0,1,La balsa,Los Gatos,1967,"6 ContributorsLa Balsa Lyrics[Letra de ""La Bal..."
1,2,Muchacha (Ojos de papel),Almendra,1970,16 ContributorsTranslationsEnglishMuchacha (Oj...
2,3,Rasguña las piedras,Sui Generis,1973,10 ContributorsRasguña las Piedras Lyrics[Letr...


In [ ]:
# Modify column names: lower case, replace spaces and hyphens with underscores, and remove non-alphanumeric characters
df.columns = (
    df.columns.str.lower()  # Convert to lowercase
    .str.replace(" ", "_")  # Replace spaces with underscores
    .str.replace("-", "_")  # Replace hyphens with underscores
    .str.replace(r"[^a-zA-Z0-9_]", "", regex=True)  # Remove non-alphanumeric characters
)  
df.columns

Index(['posicin', 'ttulo', 'artista', 'ao', 'lyrics'], dtype='object')

In [5]:
# Initialize the NER pipeline with the Spanish BERT model
ner_pipeline = pipeline(
    "ner",
    model="mrm8488/bert-spanish-cased-finetuned-ner",
    aggregation_strategy="simple",  # To merge subwords belonging to the same entity
)


# Function to extract entities from text
def extract_entities(text):
    # Skip processing if text is NaN
    if pd.isna(text):
        return {}

    # Process the text through the NER model
    try:
        # Handle long texts by splitting into chunks (BERT models typically have a 512 token limit)
        max_length = 450  # Being conservative to account for tokenization differences

        if len(text.split()) > max_length:
            # Split into chunks with some overlap
            words = text.split()
            chunks = [
                " ".join(words[i : i + max_length])
                for i in range(0, len(words), max_length - 50)
            ]

            # Process each chunk
            all_entities = []
            for chunk in chunks:
                chunk_entities = ner_pipeline(chunk)
                all_entities.extend(chunk_entities)

            entities = all_entities
        else:
            entities = ner_pipeline(text)

        # Group entities by their type
        entity_groups = {}
        for entity in entities:
            entity_type = entity["entity_group"]
            if entity_type not in entity_groups:
                entity_groups[entity_type] = []
            entity_groups[entity_type].append(entity["word"])

        return entity_groups
    except Exception as e:
        print(f"Error processing text: {e}")
        return {}


# Apply NER to the dataframe and create new columns for each entity type
def apply_ner_to_dataframe(df):
    # Create a copy of the dataframe to avoid modifying the original
    df_with_ner = df.copy()

    # Process each row and extract entities
    print("Extracting named entities from lyrics...")
    entity_results = []

    for text in tqdm(df_with_ner["lyrics"]):
        entity_results.append(extract_entities(text))

    # Find all unique entity types
    entity_types = set()
    for result in entity_results:
        entity_types.update(result.keys())

    # Create a new column for each entity type
    for entity_type in entity_types:
        column_name = f"NER_{entity_type}"
        df_with_ner[column_name] = [
            result.get(entity_type, []) for result in entity_results
        ]

    return df_with_ner


# Apply the function to process the dataframe
df = apply_ner_to_dataframe(df)
df.head(10)

Some weights of the model checkpoint at mrm8488/bert-spanish-cased-finetuned-ner were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Extracting named entities from lyrics...


  9%|▉         | 9/100 [00:02<00:35,  2.60it/s]

Error processing text: The size of tensor a (673) must match the size of tensor b (512) at non-singleton dimension 1


 11%|█         | 11/100 [00:03<00:26,  3.36it/s]

...

Error processing text: The size of tensor a (536) must match the size of tensor b (512) at non-singleton dimension 1


 91%|█████████ | 91/100 [00:24<00:03,  2.90it/s]

Error processing text: The size of tensor a (570) must match the size of tensor b (512) at non-singleton dimension 1


100%|██████████| 100/100 [00:28<00:00,  3.57it/s]


,posicin,ttulo,artista,ao,lyrics,NER_MISC,NER_PER,NER_LOC,NER_ORG
0,1,La balsa,Los Gatos,1967,"6 ContributorsLa Balsa Lyrics[Letra de ""La Bal...",[La Balsa],"[Ly, ##ric]",[],[]
1,2,Muchacha (Ojos de papel),Almendra,1970,16 ContributorsTranslationsEnglishMuchacha (Oj...,"[Ojos de Pap, Mucha, ##cha, Ojos de Papel]",[##uchacha],[],[]
2,3,Rasguña las piedras,Sui Generis,1973,10 ContributorsRasguña las Piedras Lyrics[Letr...,"[##asguña las Piedra, Lyric, "" Rasguña las Pie...","[Char, ##ly García, Nito Mestre, García, Mestr...",[],[]
3,4,De música ligera,Soda Stereo,1990,21 ContributorsTranslationsEnglishDe Música Li...,"[##e Música Ligera Ly, De Música Ligera, Canci...","[Gui, Gustavo Cerati]",[],[Soda Stereo]
4,5,Ji ji ji,Patricio Rey y sus Redonditos de Ricota,1986,NaN,[],[],[],[]
5,6,Sólo le pido a Dios,León Gieco,1978,4 ContributorsSólo le Pido a Dios Lyrics[Letra...,[Sólo Le Pido a Dios],[Lyrics [],[],[]
6,7,Presente (El momento en que estás),Vox Dei,1970,3 ContributorsPresente (El Momento En Que Está...,[],"[Ly, ##rics]",[],[]
7,8,Seminare,Serú Girán,1978,12 ContributorsTranslationsEnglishSeminare Lyr...,"[Seminare, Dios]",[##ric],[],[]
8,9,Y dale alegría a mi corazón,Fito Páez,1990,4 ContributorsY Dale Alegría a mi Corazón Lyri...,"[Dale Alegría a mi Corazón, Y Dale Alegría a M...","[Lyrics, Fito Páez, Luis Alberto Spinetta, Fit...",[],[]
9,10,Matador,Los Fabulosos Cadillacs,1993,13 ContributorsMatador LyricsLa canción “Matad...,[],[],[],[]


In [6]:
df["NER_LOC"].value_counts()

NER_LOC
[]                                                                                  80
[Buenos Aires]                                                                       2
[Baby, Baby, Baby, Baby, Baby, Baby, Baby, Baby, Baby, Baby, Baby]                   1
[Lyrics]                                                                             1
[Dios, Chapita, Palomar, Chapita, Palomar]                                           1
[Ca, Ca]                                                                             1
[Casa Rosada]                                                                        1
[Corrientes, La Paz]                                                                 1
[Haedo, ##te]                                                                        1
[Sar, ##miento, Esmeralda]                                                           1
[Calle, ár, Abasto, Abast, Ab, Abast, Ab, Calle, ár, Abasto, Abasto, Calle, ár]      1
[Ly]                               

In [11]:
# Import necessary libraries
import pandas as pd
import spacy
from collections import defaultdict

# Load Spanish large model
nlp = spacy.load("es_core_news_lg")


# Function to extract entities from text
def extract_entities(text):
    if pd.isna(text):
        return {}

    try:
        # Process the text with spaCy
        doc = nlp(text)

        # Group entities by type, removing duplicates
        entities = defaultdict(set)
        for ent in doc.ents:
            entities[ent.label_].add(ent.text)

        # Convert sets to comma-separated strings
        return {label: ", ".join(texts) for label, texts in entities.items()}
    except Exception as e:
        print(f"Error processing text: {e}")
        return {}


# Apply NER to the DataFrame
def apply_ner_to_dataframe(df):
    # Create a copy to avoid modifying the original
    result_df = df.copy()

    # Process each row in the lyrics column
    entities_series = result_df["lyrics"].apply(extract_entities)[[2]]

    # Find all unique entity types
    all_entity_types = set()
    for entity_dict in entities_series:
        all_entity_types.update(entity_dict.keys())

    # Create columns for each entity type
    for entity_type in all_entity_types:
        column_name = f"NER_{entity_type}"
        result_df[column_name] = entities_series.apply(lambda x: x.get(entity_type, ""))

    return result_df


# Apply the function to process the dataframe
df = apply_ner_to_dataframe(df)

# Display the first few rows to see the new entity columns
df.head(10)

,posicin,ttulo,artista,ao,lyrics,NER_MISC,NER_PER,NER_LOC,NER_ORG
0,1,La balsa,Los Gatos,1967,"6 ContributorsLa Balsa Lyrics[Letra de ""La Bal...",NaN,NaN,NaN,[Construiré]
1,2,Muchacha (Ojos de papel),Almendra,1970,16 ContributorsTranslationsEnglishMuchacha (Oj...,NaN,NaN,NaN,[]
2,3,Rasguña las piedras,Sui Generis,1973,10 ContributorsRasguña las Piedras Lyrics[Letr...,"Verso 2, ContributorsRasguña, Apoyo mis espald...","Puente, Nito Mestre, Mestre, García","Rasguña las Piedras, Mestre, Coro",[]
3,4,De música ligera,Soda Stereo,1990,21 ContributorsTranslationsEnglishDe Música Li...,NaN,NaN,NaN,[ContributorsTranslationsEnglishDe Música Lige...
4,5,Ji ji ji,Patricio Rey y sus Redonditos de Ricota,1986,NaN,NaN,NaN,NaN,[]
5,6,Sólo le pido a Dios,León Gieco,1978,4 ContributorsSólo le Pido a Dios Lyrics[Letra...,NaN,NaN,NaN,[]
6,7,Presente (El momento en que estás),Vox Dei,1970,3 ContributorsPresente (El Momento En Que Está...,NaN,NaN,NaN,[]
7,8,Seminare,Serú Girán,1978,12 ContributorsTranslationsEnglishSeminare Lyr...,NaN,NaN,NaN,"[Coro 1, Coro 2, Coro 1]"
8,9,Y dale alegría a mi corazón,Fito Páez,1990,4 ContributorsY Dale Alegría a mi Corazón Lyri...,NaN,NaN,NaN,[uh-uh]
9,10,Matador,Los Fabulosos Cadillacs,1993,13 ContributorsMatador LyricsLa canción “Matad...,NaN,NaN,NaN,"[El Matador, Matador!, Matador!, Matador!, Cor..."


In [12]:
df["NER_PER"].value_counts()

NER_PER
Puente, Nito Mestre, Mestre, García    1
Name: count, dtype: int64